# Sequential Multi-Agent Workflow

This notebook creates a **sequential workflow agent** that orchestrates the two agents
from notebooks 01 and 02:

1. **enterprise-knowledge-agent** — HR policies, Bing search, AI Search (fiscal reports)
2. **med-diagnostic-agent** — LOF outlier analysis, scatter plots, treatment recommendations

Two approaches are demonstrated:
- **Approach A**: `WorkflowAgentDefinition` (declarative YAML via `azure-ai-projects` SDK)
- **Approach B**: `agent-framework` SDK with `SequentialBuilder` (pro-code)

References:
- [Foundry Workflows concept](https://learn.microsoft.com/azure/foundry/agents/concepts/workflow)
- [WorkflowAgentDefinition API](https://learn.microsoft.com/python/api/azure-ai-projects/azure.ai.projects.models.workflowagentdefinition)
- [Agent Framework sequential orchestration](https://learn.microsoft.com/agent-framework/workflows/orchestrations/sequential)
- [Agents in Workflows tutorial](https://learn.microsoft.com/agent-framework/workflows/agents-in-workflows)

### Import Libraries & Authenticate

In [2]:
import os
import json
import asyncio
from typing import Any, List, Dict
from dotenv import load_dotenv

# azure-ai-projects v2 SDK — provides AIProjectClient and agent definitions.
# WorkflowAgentDefinition is the key class for Approach A (declarative YAML).
from azure.ai.projects import AIProjectClient
import azure.ai.projects as projectslib
from azure.ai.projects.models import (
    WorkflowAgentDefinition,
    PromptAgentDefinition,
)

# OpenAI types for feeding function-call results back to the agent
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print('please login first')

Environment and authentication OK


### Create Clients

In [3]:
# ---------------------------------------------------------------------------
# AIProjectClient with allow_preview=True
# ---------------------------------------------------------------------------
# WorkflowAgentDefinition requires the preview feature flag:
#   Foundry-Features: WorkflowAgents=V1Preview
# Setting allow_preview=True on the client adds this header automatically.
# Without it, create_version() returns:
#   HttpResponseError: preview_feature_required
# ---------------------------------------------------------------------------
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
    allow_preview=True,  # Required for WorkflowAgentDefinition (preview feature)
)
openai_client = project_client.get_openai_client()

print(f"azure-ai-projects version: {projectslib.__version__}")

azure-ai-projects version: 2.1.0


### Verify existing agents

The two agents must already exist from notebooks 01 and 02.
Run those notebooks first if you see errors below.

In [4]:
KNOWLEDGE_AGENT_NAME = "enterprise-knowledge-agent"
DIAGNOSTIC_AGENT_NAME = "med-diagnostic-agent"

# Verify both agents exist
knowledge_agent = project_client.agents.get(agent_name=KNOWLEDGE_AGENT_NAME)
print(f"agent > {knowledge_agent.name} (id: {knowledge_agent.id})")

diagnostic_agent = project_client.agents.get(agent_name=DIAGNOSTIC_AGENT_NAME)
print(f"agent > {diagnostic_agent.name} (id: {diagnostic_agent.id})")

agent > enterprise-knowledge-agent (id: enterprise-knowledge-agent)
agent > med-diagnostic-agent (id: med-diagnostic-agent)


---
## Approach A: Declarative Workflow (YAML + `WorkflowAgentDefinition`)

Define a sequential workflow in CSDL YAML and deploy it as a versioned agent
via `project_client.agents.create_version()` with `WorkflowAgentDefinition`.

The YAML uses `InvokeAzureAgent` actions to call the existing agents by name.
Each agent retains all its tools from notebooks 01 and 02:

| Step | Agent | Tools available |
|------|-------|----------------|
| 1 | **med-diagnostic-agent** | Code Interpreter (CSV data), File Search (treatment guidelines) |
| 2 | **enterprise-knowledge-agent** | **Bing web search**, File Search (HR policies), AI Search (fiscal reports), fetch_datetime |

The prompt must explicitly request **web research** so the knowledge agent
activates its Bing tool — otherwise it defaults to only the indexed documents.

In [39]:
# ---------------------------------------------------------------------------
# Sequential Workflow YAML (CSDL format)
# ---------------------------------------------------------------------------
# Uses the trigger-based CSDL schema that the Foundry API validates.
# Requires allow_preview=True on AIProjectClient (workflow agents are preview).
# Order: diagnostic agent first → knowledge agent second.
# Reference: https://learn.microsoft.com/azure/foundry/agents/concepts/workflow
# ---------------------------------------------------------------------------

WORKFLOW_YAML = f"""
kind: Workflow
trigger:
  kind: OnConversationStart
  id: clinical_diagnostic_research_workflow
  actions:
    - kind: InvokeAzureAgent
      id: diagnostic_analysis
      displayName: Diagnostic analysis
      agent:
        name: {DIAGNOSTIC_AGENT_NAME}
      output:
        autoSend: true
    - kind: InvokeAzureAgent
      id: treatment_research
      displayName: Treatment research
      agent:
        name: {KNOWLEDGE_AGENT_NAME}
      output:
        autoSend: true
""".strip()

print(WORKFLOW_YAML)

kind: Workflow
trigger:
  kind: OnConversationStart
  id: clinical_diagnostic_research_workflow
  actions:
    - kind: InvokeAzureAgent
      id: diagnostic_analysis
      displayName: Diagnostic analysis
      agent:
        name: med-diagnostic-agent
      output:
        autoSend: true
    - kind: InvokeAzureAgent
      id: treatment_research
      displayName: Treatment research
      agent:
        name: enterprise-knowledge-agent
      output:
        autoSend: true


In [40]:
# ---------------------------------------------------------------------------
# Deploy the workflow as a versioned agent
# ---------------------------------------------------------------------------
# WorkflowAgentDefinition takes a single `workflow` string (the CSDL YAML).
# The API validates the YAML and creates a new agent version.
# This workflow agent can then be invoked like any other agent via
# openai_client.responses.create() with an agent_reference.
# ---------------------------------------------------------------------------
WORKFLOW_AGENT_NAME = "clinical-diagnostic-research-workflow"

workflow_agent = project_client.agents.create_version(
    agent_name=WORKFLOW_AGENT_NAME,
    definition=WorkflowAgentDefinition(
        workflow=WORKFLOW_YAML,
    ),
    description="Sequential workflow: diagnostic agent → knowledge agent",
)
print(f"workflow agent > {workflow_agent.name} (id: {workflow_agent.id}, version: {workflow_agent.version})")

workflow agent > clinical-diagnostic-research-workflow (id: clinical-diagnostic-research-workflow:1, version: 1)


In [41]:
# ---------------------------------------------------------------------------
# Test the workflow agent via the Responses API
# ---------------------------------------------------------------------------
# How the sequential workflow works with autoSend:
#
#   User prompt → [diagnostic agent] → agent 1 output → [knowledge agent] → final output
#                                       ↑ autoSend ↑
#
# 1. The user prompt goes to the FIRST agent (med-diagnostic-agent) only.
# 2. autoSend: true means agent 1's complete output is automatically forwarded
#    as the input message to agent 2 (enterprise-knowledge-agent).
# 3. Agent 2 sees agent 1's findings + open questions and uses its own tools
#    (Bing web search, file_search, AI Search) to answer them.
# 4. The final response.output_text contains agent 2's output.
#
# IMPORTANT prompt design for sequential workflows:
#   - The prompt should focus on the FIRST agent's task (diagnostic analysis).
#   - Do NOT mention "Step 2" or the second agent — the first agent will try
#     to handle everything itself if you do.
#   - Instead, ask the first agent to end with "open questions" that naturally
#     trigger the second agent's tools (e.g., "search the web", "check policies").
# ---------------------------------------------------------------------------

# Create a fresh conversation for the workflow
conversation = openai_client.conversations.create()
print(f"conversation > created (id: {conversation.id})")

# Point the Responses API at our deployed workflow agent
AGENT_REFERENCE = {"name": WORKFLOW_AGENT_NAME, "type": "agent_reference"}

# The prompt targets the diagnostic agent (step 1) with patient context.
# It focuses on data analysis tasks the diagnostic agent can perform,
# and asks for "open questions" that the knowledge agent (step 2) will
# pick up via autoSend — triggering Bing search, file_search, and AI Search.
response = openai_client.responses.create(
    conversation=conversation.id,
    input=(
        "Patient case: PATIENT_ID 42 is a 38-year-old male diagnosed with Stage III "
        "non-small cell lung cancer (NSCLC). Tumor size is 5.2 cm. The patient is a "
        "non-smoker with no family history of cancer, which is unusual for this stage.\n\n"
        "Using the lung cancer CSV datasets, perform the following diagnostic analysis:\n"
        "1. Load both CSV files and join on PATIENT_ID to get the full patient profile\n"
        "2. Normalize numeric features and run LOF outlier detection on the full cohort\n"
        "3. Report whether patient 42 is an outlier and list the top deviating features\n"
        "4. Check the treatment guidelines for this patient's cancer stage\n"
        "5. Assess the patient's risk profile and identify what makes this case unusual\n"
        "6. End your report with a list of open questions that need further research, "
        "such as:\n"
        "   - What are the latest clinical trials for Stage III NSCLC in young non-smokers?\n"
        "   - What does the web say about treatment advances for this risk profile?\n"
        "   - What are the company policies on clinical trial eligibility and insurance coverage?\n"
        "   - What are the recommended first-line treatments for this cancer stage?"
    ),
    extra_body={"agent_reference": AGENT_REFERENCE},
)

# response.output_text contains the LAST agent's output (knowledge agent),
# which includes the integrated findings from both agents.
print(f"\n{'='*60}")
print(f"Workflow response (both agents combined):\n")
print(response.output_text)

conversation > created (id: conv_a2c2fe495c53101c00zFKU5P1e5qzEPDTCoAnQZ6JP5FjAdFBV)

Workflow response (both agents combined):

I have received two files: the clinical features dataset and the symptoms dataset for lung cancer patients. 

Please provide the PATIENT_ID of the patient you want me to analyze so I can start the detailed analysis, including profile extraction, outlier detection, feature importance evaluation, and visualization.Here is a summary of key HR policies you might be interested in:

1. Remote Work Policy:
- Eligibility: After 3 months tenure, applicable to roles not requiring constant on-site presence.
- Work Hours: Same core hours as on-site employees unless otherwise agreed.
- Communication: Use official channels; be available during core hours.
- Data Security: Use company-approved devices and follow secure authentication.
- Performance: Assessed by deliverable quality, responsiveness, collaboration.

2. General HR Policy:
- Equal opportunity employer with no to

---
## Approach B: Pro-Code Workflow (`agent-framework` SDK)

Use `FoundryChatClient` + `SequentialBuilder` from the `agent-framework` package
to build a sequential workflow programmatically. This gives full control over
agent instructions, streaming, and inter-agent data passing.

> **Key difference from Approach A**: `FoundryChatClient.as_agent()` creates
> LLM-only agents without Foundry tools (no Bing, no code interpreter). These
> agents reason and synthesize using their instructions and the LLM's knowledge.
> For tool-equipped workflows, use Approach A with the deployed Foundry agents.

References:
- [SequentialBuilder](https://learn.microsoft.com/agent-framework/workflows/orchestrations/sequential)
- [Agents in Workflows](https://learn.microsoft.com/agent-framework/workflows/agents-in-workflows)
- [Workflow as Agent](https://learn.microsoft.com/agent-framework/workflows/as-agents)

In [42]:
# ---------------------------------------------------------------------------
# Approach B: Pro-code workflow with agent-framework SDK
# ---------------------------------------------------------------------------
# FoundryChatClient wraps the same Foundry project endpoint into the
# agent-framework's Agent abstraction, allowing SequentialBuilder to
# chain agents programmatically.
#
# Known issue: agent-framework 1.2.2 has an import incompatibility with
# agent-framework-foundry on Python 3.14. If you see:
#   ImportError: cannot import name 'AgentMiddlewareLayer'
# upgrade both packages:
#   pip install --upgrade agent-framework agent-framework-foundry
# ---------------------------------------------------------------------------
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder
from azure.identity import AzureCliCredential

# FoundryChatClient shares the same project endpoint and model
chat_client = FoundryChatClient(
    project_endpoint=settings.project_endpoint,
    model=settings.model_deployment_name,
    credential=AzureCliCredential(),
)

print(f"FoundryChatClient ready (model: {settings.model_deployment_name})")

FoundryChatClient ready (model: gpt-4.1-mini)


In [43]:
# ---------------------------------------------------------------------------
# Build a "Diagnose → Research" sequential workflow
# ---------------------------------------------------------------------------
# Clinical workflow mirrors real practice:
#   Step 1 – DiagnosticAgent reasons about the patient's data and produces a
#            structured clinical summary with risk assessment and open questions.
#   Step 2 – ResearchAgent takes those findings and produces evidence-based
#            treatment recommendations, citing guidelines and latest practices.
#
# NOTE: FoundryChatClient.as_agent() creates LLM-only agents (no Bing/code
# interpreter). For tool-equipped agents, use Approach A with the deployed
# Foundry agents which retain their full tool configurations.
# ---------------------------------------------------------------------------

# Step 1: Diagnostic analysis agent (runs first)
diagnostic_wf_agent = chat_client.as_agent(
    name="DiagnosticAgent",
    instructions=(
        "You are a medical diagnostic analyst specializing in lung cancer.\n"
        "Given a patient case, produce a structured **Clinical Summary**:\n"
        "1. **Patient Profile**: age, cancer stage, tumor size, key clinical features\n"
        "2. **Risk Assessment**: identify which features are unusual compared to "
        "typical patients (e.g., unusually young age for advanced stage, atypical "
        "symptom combinations, outlier tumor size)\n"
        "3. **Preliminary Diagnosis**: assess severity and urgency\n"
        "4. **Open Questions**: list 3-5 specific questions for a research agent to "
        "investigate, such as:\n"
        "   - What are the recommended first-line treatments for Stage X lung cancer?\n"
        "   - Are there clinical trials for patients with this risk profile?\n"
        "   - What is the expected prognosis for this combination of factors?\n"
        "Be specific — the research agent will use your questions to search for answers."
    ),
)

# Step 2: Knowledge & treatment research agent (runs second)
research_wf_agent = chat_client.as_agent(
    name="ResearchAgent",
    instructions=(
        "You are a medical research assistant at Contoso Health.\n"
        "You receive a Clinical Summary with Open Questions from a diagnostic agent.\n\n"
        "Your job:\n"
        "1. **Answer each Open Question** with specific, evidence-based information "
        "citing treatment guidelines (NCCN, ASCO, ESMO) and recent clinical practices.\n"
        "2. **Treatment Recommendations**: for the patient's cancer stage and risk "
        "profile, recommend specific treatment protocols (surgery, chemotherapy "
        "regimens, immunotherapy, radiation) with expected outcomes.\n"
        "3. **Clinical Trial Eligibility**: note any ongoing trial categories that "
        "match the patient's profile.\n"
        "4. **Company Policy Notes**: mention that Contoso Health requires a second "
        "opinion for Stage III+ cases and covers clinical trial participation.\n\n"
        "Produce a final **Integrated Clinical Report** that combines the diagnostic "
        "findings with your research into one actionable document for the care team."
    ),
)

# Wire the sequential workflow: Diagnostic → Research
workflow = SequentialBuilder(
    participants=[diagnostic_wf_agent, research_wf_agent]
).build()

workflow_as_agent = workflow.as_agent(name="Clinical Diagnostic Research Pipeline")

print("Sequential workflow built: DiagnosticAgent → ResearchAgent")

Sequential workflow built: DiagnosticAgent → ResearchAgent


In [44]:
# --- Run the clinical diagnostic → research workflow with streaming ---
async def run_workflow():
    query = (
        "Patient case: PATIENT_ID 42 is a 38-year-old male diagnosed with Stage III "
        "non-small cell lung cancer (NSCLC). Tumor size is 5.2 cm. The patient is a "
        "non-smoker with no family history of cancer, which is unusual for this stage.\n\n"
        "Please perform a full clinical workup:\n"
        "1. Assess the patient's risk profile and identify what makes this case unusual\n"
        "2. Research the best treatment options for Stage III NSCLC in young non-smokers\n"
        "3. Check for relevant clinical trials and company coverage policies\n"
        "4. Produce an integrated report with actionable recommendations for the care team"
    )
    print(f"Query:\n{query}")
    print("=" * 60)

    current_author = None
    async for update in workflow_as_agent.run(query, stream=True):
        # Show when a different agent starts responding
        if update.author_name and update.author_name != current_author:
            if current_author:
                print("\n" + "-" * 40)
            print(f"\n[{update.author_name}]:")
            current_author = update.author_name

        if update.text:
            print(update.text, end="", flush=True)

    print("\n" + "=" * 60)
    print("Workflow completed!")

await run_workflow()

Query:
Patient case: PATIENT_ID 42 is a 38-year-old male diagnosed with Stage III non-small cell lung cancer (NSCLC). Tumor size is 5.2 cm. The patient is a non-smoker with no family history of cancer, which is unusual for this stage.

Please perform a full clinical workup:
1. Assess the patient's risk profile and identify what makes this case unusual
2. Research the best treatment options for Stage III NSCLC in young non-smokers
3. Check for relevant clinical trials and company coverage policies
4. Produce an integrated report with actionable recommendations for the care team

[ResearchAgent]:
**Integrated Clinical Report for PATIENT_ID 42**

---

### Patient Summary
- **Diagnosis**: Stage III Non-Small Cell Lung Cancer (NSCLC)  
- **Age/Sex**: 38-year-old male  
- **Tumor size**: 5.2 cm  
- **Risk Factors**: Non-smoker, no family history of cancer (unusual for Stage III NSCLC)

---

### 1. Risk Assessment and Unusual Features

- **Age and non-smoking status**: NSCLC typically affects

### DevUI: Interactive Web Interface

Launch the workflow (and individual agents) in [DevUI](https://learn.microsoft.com/agent-framework/devui/) —
a lightweight web app for interactive testing with streaming, tracing, and multi-turn conversations.

> DevUI uses **`FoundryAgent`** to wrap the deployed Foundry agents, giving
> full tool access (code interpreter, file search, Bing search).
>
> The server runs in a **self-contained daemon thread** with its own event loop.
> This is safe for "Run All" — the thread won't block the kernel. Re-run to restart.

#### DevUI entities registered

| Entity | Type | UI | Both agents? | Images? |
|--------|------|-----|-------------|---------|
| **WorkflowBuilder** | `workflow` | Configure & Run form + pipeline graph | **Yes** — both agents chain correctly | Broken links (client-rendered timeline) |
| **Clinical Pipeline** | `agent` (via `wf.as_agent()`) | Simple text chat | **No** — only 1st agent runs (framework bug) | Stripped by Patch 5 |
| **med-diagnostic-agent** | `agent` | Simple text chat | N/A (single agent) | Stripped by Patch 5 |
| **enterprise-knowledge-agent** | `agent` | Simple text chat | N/A (single agent) | N/A (no code interpreter) |

**Recommended:** Use the **WorkflowBuilder** entity for testing the full sequential pipeline.

#### Monkey patches (cell 18)

The monkey-patch cell applies 5 runtime patches to `agent-framework` v1.0.0b260429:

| Patch | Target | What it does |
|-------|--------|-------------|
| **Patch 1** | `InProcRunnerContext.send_message` / `has_messages` | Diagnostic logging for inter-executor message delivery |
| **Patch 2a** | `WorkflowAgent._convert_workflow_events_to_agent_response` | Downloads container images → base64 data URIs (non-streaming agent chat path) |
| **Patch 2b** | `MessageMapper._convert_workflow_event` | Downloads container images → base64 data URIs (WorkflowBuilder path) |
| **Patch 3** | `AgentExecutor.from_response` | Strips `container_file_citation` annotations from messages passed between executors — fixes 400 error: `Missing required parameter: annotations[].index` |
| **Patch 4** | `MessageMapper._create_unknown_content_event` | Suppresses noisy "Warning: Unknown content type: Content" messages |
| **Patch 5** | `MessageMapper.convert_event` | Top-level intercept: strips ALL broken image refs (`sandbox:`, `/mnt/data/`, bare filenames) from streaming output using whitelist approach (only `data:` and `http` URLs pass through) |

#### Image handling

Code interpreter generates matplotlib plots saved to the sandbox filesystem. The agent references
them as `sandbox:/mnt/data/*.png` or `/mnt/data/*.png` — neither resolves in DevUI.

**Image pipeline (Patches 2a/2b):**
```
container_file_citation annotation → extract container_id + file_id
→ openai_client.containers.files.content.retrieve() → base64 encode
→ replace sandbox: URL with data:image/png;base64,... URI
```

**Limitation:** The WorkflowBuilder's Execution Timeline panel renders stored event data
client-side in React. Server-side monkey patches cannot intercept that rendering path.
Images in the timeline view remain as broken text links. The **Gradio UI in notebook 02**
is the most reliable way to see inline scatter plots.

Sample prompts for DevUI (copy-paste into the chat):

Prompt 1 (short — relies on the agent's built-in instructions to generate scatter plots):
```text
Patient case: PATIENT_ID 42

Please perform a full clinical workup:
1. Assess the patient's risk profile and identify what makes this case unusual
2. Research the best treatment options for Stage III NSCLC in young non-smokers
3. Check for relevant clinical trials and company coverage policies
4. Produce an integrated report with actionable recommendations for the care team
```

Prompt 2 (explicit — requests data analysis, scatter plot, and research):
```text
Using the lung cancer CSV datasets, analyze PATIENT_ID 42:
1. Load both CSV files and join on PATIENT_ID to get the full patient profile
2. Normalize numeric features and run LOF outlier detection on the full cohort
3. Generate a 2D scatter plot highlighting patient 42 among all patients
4. Report whether patient 42 is an outlier and list the top deviating features
5. Check the treatment guidelines for this patient's cancer stage
6. End with open questions for further research: latest clinical trials, treatment advances, company policies on trial eligibility
```

In [5]:
# ---------------------------------------------------------------------------
# Monkey-patches for agent-framework v1.0.0b260429
# ---------------------------------------------------------------------------
# Patch 1: Diagnostic logging for inter-executor message delivery
# Patch 2: Download code-interpreter images via containers API → base64 data URIs
# Patch 3: Strip container_file_citation annotations between executors (400 fix)
# ---------------------------------------------------------------------------
import re
import base64
import logging
from agent_framework._workflows._runner_context import InProcRunnerContext
from agent_framework._workflows._agent import WorkflowAgent
from agent_framework._workflows._agent_executor import AgentExecutor, AgentExecutorResponse
from agent_framework_devui._mapper import MessageMapper

_patch_logger = logging.getLogger("monkey_patch")
_patch_logger.setLevel(logging.INFO)

# ---------------------------------------------------------------------------
# Image helpers
# ---------------------------------------------------------------------------
def _download_image_b64(oc, container_id, file_id, filename):
    try:
        content = oc.containers.files.content.retrieve(file_id=file_id, container_id=container_id)
        image_bytes = content.read()
        b64 = base64.b64encode(image_bytes).decode("utf-8")
        ext = filename.rsplit(".", 1)[-1].lower()
        mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg",
                "gif": "image/gif", "svg": "image/svg+xml"}.get(ext, "image/png")
        _patch_logger.info(f"[IMAGE] Downloaded {filename} ({len(image_bytes)} bytes)")
        return f"data:{mime};base64,{b64}"
    except Exception as e:
        _patch_logger.warning(f"[IMAGE] Download failed for {filename}: {e}")
        return None

def _fetch_container_images(oc, container_id):
    images = {}
    try:
        files = oc.containers.files.list(container_id=container_id)
        for f in files:
            fname = getattr(f, "filename", "") or getattr(f, "name", "") or ""
            fid = getattr(f, "id", "")
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.svg')):
                uri = _download_image_b64(oc, container_id, fid, fname)
                if uri:
                    images[fname] = uri
                    basename = fname.rsplit("/", 1)[-1] if "/" in fname else fname
                    images[basename] = uri
    except Exception as e:
        _patch_logger.warning(f"[IMAGE] Container file list failed: {e}")
    return images

# Whitelist: strip ALL ![...](URL) where URL does NOT start with data: or http
_BROKEN_IMG_RE = re.compile(r'!?\[[^\]]*\]\((?!data:|http)[^)]*\)')
_BROKEN_HTML_RE = re.compile(r'<img[^>]*(?!data:)[^>]*>', re.IGNORECASE)

def _has_broken_refs(text):
    return bool(_BROKEN_IMG_RE.search(text)) or bool(_BROKEN_HTML_RE.search(text))

def _replace_broken_with_images(text, images):
    def _replacer(m):
        full = m.group(0)
        url_match = re.search(r'\(([^)]+)\)', full)
        if url_match:
            url = url_match.group(1)
            for fname, uri in images.items():
                if fname in url:
                    alt_match = re.search(r'\[([^\]]*)\]', full)
                    alt_text = alt_match.group(1) if alt_match else fname
                    return f"\n\n![{alt_text}]({uri})"
        return ""
    result = _BROKEN_IMG_RE.sub(_replacer, text)
    result = _BROKEN_HTML_RE.sub('', result)
    return result

def _strip_broken_refs(text):
    result = _BROKEN_IMG_RE.sub('', text)
    result = _BROKEN_HTML_RE.sub('', result)
    return result

def _collect_container_ids(output_events):
    container_ids = set()
    for evt in output_events:
        data = getattr(evt, 'data', None)
        if data is None:
            continue
        agent_resp = getattr(data, 'agent_response', None)
        if agent_resp is None:
            continue
        for msg in getattr(agent_resp, 'messages', []):
            for content in getattr(msg, 'contents', []):
                for ann in getattr(content, 'annotations', []) or []:
                    cid = getattr(ann, 'container_id', None)
                    if cid:
                        container_ids.add(cid)
    return container_ids

def _collect_container_ids_from_event(event):
    container_ids = set()
    data = getattr(event, 'data', None)
    if data is not None:
        agent_resp = getattr(data, 'agent_response', None)
        if agent_resp is not None:
            for msg in getattr(agent_resp, 'messages', []):
                for content in getattr(msg, 'contents', []):
                    for ann in getattr(content, 'annotations', []) or []:
                        cid = getattr(ann, 'container_id', None)
                        if cid:
                            container_ids.add(cid)
    return container_ids


# ---------------------------------------------------------------------------
# Patch 3: Strip container_file_citation annotations between executors
# ---------------------------------------------------------------------------
# Previous approaches (swapping __wrapped__, replacing from_response on the class)
# failed because @handler registers handlers at class definition time via a closure.
# The handler registry caches the original function reference — class-level replacement
# doesn't update the dispatch table.
#
# Reliable approach: patch _run_agent_and_emit(), which is called for ALL agent
# executions regardless of handler dispatch. Strip annotations from self._cache
# (the accumulated conversation context) before the agent runs.
def _strip_container_annotations(messages):
    for msg in messages:
        for content in getattr(msg, 'contents', []):
            anns = getattr(content, 'annotations', None)
            if anns:
                cleaned = [a for a in anns if getattr(a, 'type', None) != 'container_file_citation']
                content.annotations = cleaned if cleaned else None

_orig_run_agent_and_emit = AgentExecutor._run_agent_and_emit

async def _patched_run_agent_and_emit(self, ctx):
    # Strip container_file_citation from cache before running the agent
    _strip_container_annotations(self._cache)
    return await _orig_run_agent_and_emit(self, ctx)

AgentExecutor._run_agent_and_emit = _patched_run_agent_and_emit

# ---------------------------------------------------------------------------
# Patch 1: Diagnostic logging
# ---------------------------------------------------------------------------
_orig_send = InProcRunnerContext.send_message
async def _p_send(self, message):
    _patch_logger.info(f"[PATCH] send_message: {message.source_id} → {message.target_id}")
    return await _orig_send(self, message)
InProcRunnerContext.send_message = _p_send

_orig_has = InProcRunnerContext.has_messages
async def _p_has(self):
    r = await _orig_has(self)
    _patch_logger.info(f"[PATCH] has_messages={r}")
    return r
InProcRunnerContext.has_messages = _p_has

# ---------------------------------------------------------------------------
# Patch 2a: Inline images in WorkflowAgent text chat path (non-streaming)
# ---------------------------------------------------------------------------
_orig_convert_events = WorkflowAgent._convert_workflow_events_to_agent_response

def _patched_convert_events(self, response_id, output_events):
    result = _orig_convert_events(self, response_id, output_events)
    cids = _collect_container_ids(output_events)
    all_images = {}
    if cids:
        for cid in cids:
            all_images.update(_fetch_container_images(openai_client, cid))
    for msg in result.messages:
        for content in msg.contents:
            if hasattr(content, 'text') and content.text and _has_broken_refs(content.text):
                if all_images:
                    content.text = _replace_broken_with_images(content.text, all_images)
                else:
                    content.text = _strip_broken_refs(content.text)
    return result

WorkflowAgent._convert_workflow_events_to_agent_response = _patched_convert_events

# Streaming path — strip only
_orig_convert_updates = WorkflowAgent._convert_workflow_event_to_agent_response_updates
def _patched_convert_updates(self, response_id, event):
    updates = _orig_convert_updates(self, response_id, event)
    for u in updates:
        if hasattr(u, 'text') and u.text and _has_broken_refs(u.text):
            u.text = _strip_broken_refs(u.text)
    return updates
WorkflowAgent._convert_workflow_event_to_agent_response_updates = _patched_convert_updates

# ---------------------------------------------------------------------------
# Patch 2b: Inline images in DevUI WorkflowBuilder path
# ---------------------------------------------------------------------------
_orig_convert_wf = MessageMapper._convert_workflow_event

async def _patched_convert_wf(self, event, context):
    results = await _orig_convert_wf(self, event, context)
    has_broken = False
    for item in results:
        if hasattr(item, 'content'):
            for ci in getattr(item, 'content', []):
                if hasattr(ci, 'text') and ci.text and _has_broken_refs(ci.text):
                    has_broken = True
                    break
    if has_broken:
        cids = _collect_container_ids_from_event(event)
        all_images = {}
        if cids:
            for cid in cids:
                all_images.update(_fetch_container_images(openai_client, cid))
        for item in results:
            if hasattr(item, 'content'):
                for ci in getattr(item, 'content', []):
                    if hasattr(ci, 'text') and ci.text and _has_broken_refs(ci.text):
                        ci.text = _replace_broken_with_images(ci.text, all_images) if all_images else _strip_broken_refs(ci.text)
            if hasattr(item, 'delta') and isinstance(getattr(item, 'delta', None), str):
                if _has_broken_refs(item.delta):
                    item.delta = _strip_broken_refs(item.delta)
    return results

MessageMapper._convert_workflow_event = _patched_convert_wf

# ---------------------------------------------------------------------------
# Patch 4: Suppress "Unknown content type: Content" warnings
# ---------------------------------------------------------------------------
_orig_unknown_content = MessageMapper._create_unknown_content_event

async def _patched_unknown_content(self, content, context):
    return self._create_text_delta_event("", context)

MessageMapper._create_unknown_content_event = _patched_unknown_content

# ---------------------------------------------------------------------------
# Patch 5: Strip broken image refs from ALL DevUI output (top-level intercept)
# ---------------------------------------------------------------------------
# Patches convert_event — the single entry point for ALL events flowing to DevUI.
# After the original method produces OpenAI events, we post-process every text
# field (content.text, delta) to strip/replace broken image references.
_orig_convert_event = MessageMapper.convert_event

async def _patched_convert_event(self, raw_event, request):
    results = await _orig_convert_event(self, raw_event, request)
    for item in results:
        # Patch ResponseOutputMessage content items
        if hasattr(item, 'content'):
            for ci in getattr(item, 'content', []):
                if hasattr(ci, 'text') and ci.text and _has_broken_refs(ci.text):
                    ci.text = _strip_broken_refs(ci.text)
        # Patch text delta events
        if hasattr(item, 'delta') and isinstance(getattr(item, 'delta', None), str):
            if _has_broken_refs(item.delta):
                item.delta = _strip_broken_refs(item.delta)
        # Patch 'text' attribute directly (some event types)
        if hasattr(item, 'text') and isinstance(getattr(item, 'text', None), str):
            if _has_broken_refs(item.text):
                item.text = _strip_broken_refs(item.text)
    return results

MessageMapper.convert_event = _patched_convert_event

print("Monkey patches applied:")
print("  ✓ Patch 1: Diagnostic logging for inter-executor message delivery")
print("  ✓ Patch 2a: Inline base64 images in WorkflowAgent text chat path")
print("  ✓ Patch 2b: Inline base64 images in DevUI WorkflowBuilder path")
print("  ✓ Patch 3: Strip container_file_citation annotations between executors")
print("  ✓ Patch 4: Suppress 'Unknown content type' warnings")
print("  ✓ Patch 5: Strip broken image refs from ALL DevUI output (top-level)")
print()
print("Image whitelist: strip ALL ![...](URL) where URL is NOT data: or http")
print("Image pipeline: container_file_citation → containers API → data:image/png;base64")

Monkey patches applied:
  ✓ Patch 1: Diagnostic logging for inter-executor message delivery
  ✓ Patch 2a: Inline base64 images in WorkflowAgent text chat path
  ✓ Patch 2b: Inline base64 images in DevUI WorkflowBuilder path
  ✓ Patch 3: Strip container_file_citation annotations between executors
  ✓ Patch 4: Suppress 'Unknown content type' warnings
  ✓ Patch 5: Strip broken image refs from ALL DevUI output (top-level)

Image whitelist: strip ALL ![...](URL) where URL is NOT data: or http
Image pipeline: container_file_citation → containers API → data:image/png;base64


In [ ]:
# ---------------------------------------------------------------------------
# Launch DevUI — safe for "Run All"
# ---------------------------------------------------------------------------
# DevUI runs in a self-contained daemon thread with its own event loop.
# Unlike Approach B (LLM-only), this uses FoundryAgent to wrap the deployed
# Foundry agents — giving full tool access: code interpreter (images!),
# file search, Bing web search, AI Search, and custom functions.
#
# FoundryAgent connects to the SAME deployed agents from notebooks 01 and 02,
# so code interpreter can generate matplotlib plots and the images are returned
# via code_interpreter_call events in the Responses API.
#
# IMPORTANT: Run the monkey-patch cell (cell 18) BEFORE this cell.
# The patches fix critical issues:
#   - Patch 3: Strips container_file_citation annotations that cause 400 errors
#              when chaining agents (missing 'index' field on annotations).
#   - Patch 4: Suppresses noisy "Unknown content type" warnings.
#   - Patch 5: Strips broken sandbox:/mnt/data/ image URLs from output.
#
# Known limitation (agent-framework v1.0.0b260429):
#   wf.as_agent() ("Clinical Diagnostic Research Pipeline") only runs the
#   FIRST executor in the streaming path. The second executor is never invoked
#   due to a framework bug in Runner.run_until_convergence() async generator.
#   Use the WorkflowBuilder entity ("Configure & Run") instead.
#
# Re-running this cell stops the old server and starts a new one.
# Reference: https://learn.microsoft.com/agent-framework/devui/
# ---------------------------------------------------------------------------
import threading
import time

DEVUI_PORT = 8080

# --- Safe restart: stop previous server if running ---
if '_devui_server' in globals() and _devui_server is not None:
    _devui_server.should_exit = True
    time.sleep(1)  # give uvicorn time to release the port
    print(f"Stopped previous DevUI server on port {DEVUI_PORT}")


def _start_devui(port: int, project_endpoint: str, cred):
    """Start DevUI in a dedicated thread with its own event loop and Foundry agents."""
    import asyncio
    import uvicorn
    from agent_framework import WorkflowBuilder
    from agent_framework.foundry import FoundryAgent
    from agent_framework.devui import DevServer
    from agent_framework.observability import enable_instrumentation

    # New event loop for this thread — all asyncio objects created here
    # will belong to this loop, avoiding cross-loop errors.
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    enable_instrumentation(enable_sensitive_data=True)

    # FoundryAgent wraps the DEPLOYED agents — full tool access including
    # code interpreter (image generation), file search, Bing, and AI Search.
    #
    # allow_preview=True routes requests to the agent-specific endpoint:
    #   .../agents/{agent_name}/endpoint/protocols/openai/
    # This means the model is resolved server-side from the agent definition,
    # so no model parameter is needed in the request.
    diag = FoundryAgent(
        project_endpoint=project_endpoint,
        agent_name="med-diagnostic-agent",
        credential=cred,
        allow_preview=True,
    )

    research = FoundryAgent(
        project_endpoint=project_endpoint,
        agent_name="enterprise-knowledge-agent",
        credential=cred,
        allow_preview=True,
    )

    # Use WorkflowBuilder (not SequentialBuilder) so the first executor is
    # the FoundryAgent itself — DevUI then:
    #   1. Shows a simple text input (not structured data form)
    #   2. Shows the pipeline visualization (type="workflow")
    #   3. Shows executor_invoked / executor_completed lifecycle events
    #
    # SequentialBuilder prepends _InputToConversation which expects Message
    # objects and triggers the "Configure Workflow Inputs" structured form.
    wf = (
        WorkflowBuilder(start_executor=diag)
        .add_edge(diag, research)
        .build()
    )

    # Also create a WorkflowAgent wrapper for simple text chat.
    # DevUI classifies it as type="agent" → simple text input box.
    # The raw Workflow (wf) gives pipeline visualization but requires
    # the "Configure & Run" form (DevUI's intended workflow UX).
    # wf_agent = wf.as_agent(name="Clinical Diagnostic Research Pipeline")

    # Register both:
    #   - wf_agent: simple text chat (type="agent"), no visualization
    #   - wf: pipeline visualization (type="workflow"), structured input form
    #   - diag/research: individual agents for standalone testing
    dev_server = DevServer(port=port)
    # dev_server.set_pending_entities([wf_agent, wf, diag, research])
    dev_server.set_pending_entities([wf, diag, research])
    app = dev_server.get_app()

    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="info")
    global _devui_server
    _devui_server = uvicorn.Server(config)
    loop.run_until_complete(_devui_server.serve())


# --- Launch in daemon thread ---
_devui_thread = threading.Thread(
    target=_start_devui,
    args=(DEVUI_PORT, settings.project_endpoint, credential),
    daemon=True,
)
_devui_thread.start()

# Brief wait for server startup (non-blocking for Run All)
time.sleep(2)

print(f"DevUI running in background thread...")
print(f"Web UI:  http://localhost:{DEVUI_PORT}")
print(f"API:     http://localhost:{DEVUI_PORT}/v1/responses")
print(f"\nAgents available (with full Foundry tools):")
print(f"  - Clinical Diagnostic Research Pipeline (text chat → runs both agents)")
print(f"  - WorkflowBuilder (pipeline visualization → Configure & Run form)")
print(f"  - med-diagnostic-agent (code interpreter + file search)")
print(f"  - enterprise-knowledge-agent (Bing + file search + AI Search)")
print(f"\nRe-run this cell to restart, or restart the kernel to stop.")

INFO:     Started server process [82443]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)


DevUI running in background thread...
Web UI:  http://localhost:8080
API:     http://localhost:8080/v1/responses

Agents available (with full Foundry tools):
  - Clinical Diagnostic Research Pipeline (text chat → runs both agents)
  - WorkflowBuilder (pipeline visualization → Configure & Run form)
  - med-diagnostic-agent (code interpreter + file search)
  - enterprise-knowledge-agent (Bing + file search + AI Search)

Re-run this cell to restart, or restart the kernel to stop.


INFO:     127.0.0.1:61945 - "GET /meta HTTP/1.1" 200 OK
INFO:     127.0.0.1:61945 - "GET /v1/entities HTTP/1.1" 200 OK
INFO:     127.0.0.1:61945 - "GET /v1/entities/workflow_in_memory_workflowbuilder-a32bb43f-a0ee-4904-bd8e-89fa82a364e3_d55d831430fa4f8a9f349ffdfd8b267f/info?type=workflow HTTP/1.1" 200 OK
INFO:     127.0.0.1:61945 - "GET /v1/conversations?entity_id=workflow_in_memory_workflowbuilder-a32bb43f-a0ee-4904-bd8e-89fa82a364e3_d55d831430fa4f8a9f349ffdfd8b267f&type=workflow_session HTTP/1.1" 200 OK
INFO:     127.0.0.1:61945 - "POST /v1/conversations HTTP/1.1" 200 OK
INFO:     127.0.0.1:61952 - "POST /v1/responses HTTP/1.1" 200 OK


---
### Notes

All three approaches use the same **Diagnose → Research** pipeline order:

| Step | Agent | Approach A (declarative YAML) | Approach B (LLM-only) | DevUI (`FoundryAgent`) |
|------|-------|-------------------------------|----------------------|----------------------|
| 1 | **DiagnosticAgent** | Code Interpreter + File Search | LLM reasoning only | Code Interpreter + File Search |
| 2 | **ResearchAgent** | Bing + File Search + AI Search | LLM synthesis only | Bing + File Search + AI Search |

**Approach A** (declarative YAML via `WorkflowAgentDefinition`):
- Full tool access: Bing web search, code interpreter, file search, AI Search
- Deployed as a versioned workflow agent via `WorkflowAgentDefinition`
- Can be edited visually in the Foundry portal workflow designer
- Prompt focuses on the diagnostic agent's task; open questions trigger the knowledge agent

**Approach B** (pro-code `agent-framework` with `FoundryChatClient.as_agent()`):
- LLM-only agents — faster but no external tool access
- Full programmatic control over instructions and data flow
- Streaming with per-agent author attribution
- Ideal for rapid prototyping and instruction tuning

**DevUI** (pro-code `agent-framework` with `FoundryAgent`):
- Wraps the **deployed** Foundry agents — full tool access like Approach A
- Interactive web UI with streaming, tracing, and multi-turn conversations
- Requires monkey patches (cell 18) to work around framework bugs — see below

**Which to choose?**
- Use **Approach A** for production deployment and portal management
- Use **Approach B** for rapid prototyping and custom orchestration logic
- Use **DevUI WorkflowBuilder** for interactive testing with full tool access

---
### Known Issues & Patches (`agent-framework` v1.0.0b260429)

#### Issue 1: `wf.as_agent()` only runs the first executor (streaming bug)

The "Clinical Diagnostic Research Pipeline" text chat entity wraps the workflow via
`wf.as_agent()`. In testing, only `med-diagnostic-agent` executes — `enterprise-knowledge-agent`
is never invoked (confirmed via DevUI Traces: only 1 `executor.process` span, total ~160s but
only ~54s in the first executor — remaining time is idle/timeout).

**Root cause:** The `@handler` decorator wraps `AgentExecutor.from_response` in a closure
(`wrapper` calls `func` directly). After the first executor's `ctx.send_message()` queues a
message, the `Runner.run_until_convergence()` async generator should start a second superstep.
However, in the streaming path (`WorkflowAgent._run_stream_impl` → `workflow.run(stream=True)`),
the runner's convergence loop doesn't pick up the queued message for the next superstep.
The `message.send` trace span (0.1ms) confirms the message was queued, but no second
`executor.process` span appears. This is a framework-level async generator lifecycle bug.

**Workaround:** Use the **WorkflowBuilder** entity ("Configure & Run" form) which uses
`_execute_workflow()` in DevUI — a different code path that correctly chains both agents.

#### Issue 2: `container_file_citation` annotation 400 error (FIXED by Patch 3)

When the diagnostic agent's code interpreter generates a scatter plot, the response includes
`container_file_citation` annotations with `container_id` and `file_id`. When these annotations
are forwarded to the second agent as input (via `AgentExecutor.from_response`), the Foundry API
rejects them with:

```
400 - Missing required parameter: 'input[1].content[0].annotations[1].index'
```

The `agent-framework` doesn't include the required `index` field when serializing annotations.

**Fix (Patch 3):** Monkey-patch `AgentExecutor.from_response` to strip all
`container_file_citation` annotations from messages before passing to the next executor.
The second agent can't access the first agent's sandbox container anyway.

**Note:** The `@handler` decorator captures `func` in a closure, so swapping `__wrapped__`
doesn't work. Patch 3 replaces the **entire function** on the class and copies `_handler_spec`
metadata so the executor's handler registry still recognizes it.

#### Issue 3: Broken image URLs in DevUI output

Code interpreter generates matplotlib plots saved to `sandbox:/mnt/data/*.png` or
`/mnt/data/*.png`. Neither URL scheme resolves in DevUI — they render as broken links
or trigger 404 errors (e.g., `http://localhost:8080/mnt/data/patient_42.png`).

**Fix (Patches 2a, 2b, 5):** Three layers of image handling:
- **Patch 2a/2b:** When `container_file_citation` annotations provide a `container_id`,
  download images via `openai_client.containers.files.content.retrieve()` and replace
  sandbox URLs with `data:image/png;base64,...` inline URIs
- **Patch 5:** Top-level whitelist intercept on `MessageMapper.convert_event` —
  strips ALL `![...](URL)` where URL doesn't start with `data:` or `http`

**Limitation:** The WorkflowBuilder's Execution Timeline panel renders stored event data
client-side in React. Server-side monkey patches cannot intercept that rendering path.
For reliable inline images, use the **Gradio UI in notebook 02**.

#### Issue 4: "Warning: Unknown content type: Content" (FIXED by Patch 4)

DevUI's `MessageMapper._create_unknown_content_event` emits warning text for `Content`
objects it doesn't know how to render (e.g., code_interpreter tool call/result Content).
These are harmless but noisy — pollute the chat output with repeated warning lines.

**Fix (Patch 4):** Replace `_create_unknown_content_event` with a no-op that returns
an empty text delta.

---
### Test Results (Playwright MCP automated testing, 2026-05-04)

| Test | WorkflowBuilder (Configure & Run) | Clinical Pipeline (wf.as_agent) |
|------|----------------------------------|-------------------------------|
| Both agents run | **PASS** | **FAIL** — only 1st agent (framework bug) |
| No 400 annotation error | **PASS** (Patch 3) | **PASS** (Patch 3) |
| No "Unknown content type" warnings | **PASS** (Patch 4) | **PASS** (Patch 4) |
| Broken image URLs stripped | **PARTIAL** — timeline is client-rendered | **PASS** (Patch 5) |
| Inline base64 images | Not visible (timeline client-side) | N/A (2nd agent never runs) |
| "Unable to process request" | Not seen | Seen intermittently (related to Issue 1) |

### (Optional) Cleanup

In [46]:
# # Delete the workflow agent version
# project_client.agents.delete_version(
#     agent_name=workflow_agent.name,
#     agent_version=workflow_agent.version,
# )
# print("Workflow agent version deleted.")